In [101]:
import os
from typing import Tuple, Optional
from fastapi import FastAPI, HTTPException, Body, Header
from fastapi.responses import JSONResponse
import geopandas as gpd
import networkx as nx
import numpy as np
from shapely.geometry import Point, LineString
from shapely.ops import linemerge
from scipy.spatial import cKDTree
from pyproj import Transformer
from joblib import dump, load
from tqdm import tqdm
import math
from matplotlib import pyplot as plt

In [102]:
APP = FastAPI(title="Tehran Routing Service")
# ---- تنظیمات ----
DATA_SHP = os.environ.get("ROADS_SHP", r"C:\Users\Amir\Desktop\SDSS-Project\Data\Roads\Tehran_Roads_line.shp")
TARGET_EPSG = int(os.environ.get("TARGET_EPSG", "32639"))  # UTM zone 39N
ADMIN_TOKEN = os.environ.get("ADMIN_TOKEN", "change-me")   # برای /reload

CACHE_DIR = os.environ.get("CACHE_DIR", "data/cache")
os.makedirs(CACHE_DIR, exist_ok=True)
CACHE_GRAPH = os.path.join(CACHE_DIR, "graph.pkl")
CACHE_NODES = os.path.join(CACHE_DIR, "nodes.npy")
CACHE_NODEIDS = os.path.join(CACHE_DIR, "node_ids.npy")
CACHE_TRANSFORMER = os.path.join(CACHE_DIR, "transformer.pkl")

# ---- متغیرهای سراسری ----
G: Optional[nx.Graph] = None
NODE_XY: Optional[np.ndarray] = None     # [[x,y], ...] in TARGET_EPSG
NODE_IDS: Optional[np.ndarray] = None    # [node_id, ...]
TREE: Optional[cKDTree] = None
GEODETIC_TO_TARGET: Optional[Transformer] = None   # 4326 -> TARGET_EPSG

In [103]:
def _build_transformer():
    # WGS84 (Leaflet lat/lng) -> UTM 32639
    return Transformer.from_crs(4326, TARGET_EPSG, always_xy=True)
# _build_transformer()

In [104]:
def _load_shapefile(path: str) -> gpd.GeoDataFrame:
    gdf = gpd.read_file(path)
    if gdf.crs is None:
        raise RuntimeError("Shapefile has no CRS. Add a .prj or set crs before.")
    gdf = gdf.to_crs(TARGET_EPSG)
    # پاکسازی هندسه‌های خالی
    gdf = gdf[~gdf.geometry.is_empty & gdf.geometry.notnull()].copy()
    # Optional: explode multiline to singleparts for cleaner graph
    gdf = gdf.explode(index_parts=False, ignore_index=True)
    # print(gdf.geometry[2])
    return gdf
# gdf = _load_shapefile(r"C:\Users\Amir\Desktop\SDSS-Project\Data\Roads\Tehran_Roads_line.shp")


In [105]:
def _gdf_to_graph(
    gdf: gpd.GeoDataFrame,
    use_tqdm: bool = True,
    progress_cb=None,          # تابع اختیاری: progress_cb(done, total)
    progress_every: int = 500  # هر چند رکورد یک بار کال‌بک صدا بخورد
) -> nx.Graph:
    """
    از LineStringها گراف می‌سازد و پیشرفت را با tqdm یا کال‌بک گزارش می‌دهد.
    """
    G = nx.Graph()
    default_speed_kmh = 40.0

    total = len(gdf.geometry)
    iterator = enumerate(gdf.geometry)

    # اگر tqdm نصب است و فعال خواستی، بپیچ داخل tqdm
    if use_tqdm and tqdm is not None:
        iterator = tqdm(iterator, total=total, desc="Building road graph", unit="feat")

    for idx, geom in iterator:
        if geom is None or geom.is_empty:
            # گزارش پیشرفت با کال‌بک (اگر tqdm نداریم/نمی‌خواهیم)
            if progress_cb and (not use_tqdm or tqdm is None) and (idx % progress_every == 0):
                progress_cb(idx, total)
            continue

        if not isinstance(geom, LineString):
            # اگر MultiLineString باقی مانده، سعی در merge
            try:
                geom = linemerge(geom)
                if not isinstance(geom, LineString):
                    if progress_cb and (not use_tqdm or tqdm is None) and (idx % progress_every == 0):
                        progress_cb(idx, total)
                    continue
            except Exception:
                if progress_cb and (not use_tqdm or tqdm is None) and (idx % progress_every == 0):
                    progress_cb(idx, total)
                continue
            
        coords = list(geom.coords)
        for i, _ in enumerate(coords):
            if i == len(coords) - 1:
                break
            start = coords[i]; end = coords[i+1]
            u = (round(start[0], 3), round(start[1], 3))
            v = (round(end[0], 3), round(end[1], 3))

            length_m = float(math.sqrt((v[0] - u[0])**2 + (v[1] - u[1])**2))
            # speed_kmh = default_speed_kmh
            # اگر فیلد کلاس/سرعت داری، اینجا مقدار بده:
            # if 'class' in gdf.columns:
            #     cls = gdf.at[idx, 'class']
            #     speed_kmh = 90 if cls in ('motorway','trunk') else 60 if cls in ('primary','secondary') else 30

            if u not in G: G.add_node(u, x=u[0], y=u[1])
            if v not in G: G.add_node(v, x=v[0], y=v[1])

            G.add_edge(u, v, length_m=length_m, geometry=LineString([u, v]))

            # اگر tqdm نداریم، هر progress_every رکورد یک‌بار کال‌بک بزن
            if progress_cb and (not use_tqdm or tqdm is None) and (idx % progress_every == 0):
                progress_cb(idx, total)

    # مرحلهٔ وزن‌دهی یال‌ها (خارج از حلقه، سریع است)
    # _add_costs_to_edges(G)

    # در پایان، 100% را اعلام کن
    if progress_cb and (not use_tqdm or tqdm is None):
        progress_cb(total, total)

    return G
# G = _gdf_to_graph(gdf)


In [106]:

# nx.draw(G, with_labels=True, node_color='lightblue', edge_color='gray')
# plt.show()

In [ ]:
def _prepare_kdtree(G: nx.Graph):
    nodes = np.array([(d["x"], d["y"]) for n, d in G.nodes(data=True)], dtype=float)
    node_ids = np.array(list(G.nodes()))
    tree = cKDTree(nodes)
    return nodes, node_ids, tree

def _project_latlng(lng: float, lat: float) -> Tuple[float, float]:
    x, y = GEODETIC_TO_TARGET.transform(lng, lat)
    return x, y

def _nearest_node_id(lng: float, lat: float):
    x, y = GEODETIC_TO_TARGET.transform(lng, lat)  # (lon,lat) -> UTM
    _, idx = TREE.query([x, y])
    node_id = NODE_IDS[idx]
    # اگر احیاناً نوعش ndarray شد، به tuple تبدیل کن
    if hasattr(node_id, "tolist"):
        node_id = tuple(node_id.tolist())
    elif isinstance(node_id, np.ndarray):
        node_id = tuple(node_id)
    return node_id, idx

In [108]:
def _route(u_id, v_id):
    # Dijkstra
    try:
        path = nx.shortest_path(G, source=u_id, target=v_id)
        # به LineString تبدیل کن
        coords = [(G.nodes[n]["x"], G.nodes[n]["y"]) for n in path]
        line = LineString(coords)
        # GeoJSON ساده
        return {
            "type": "Feature",
            "geometry": {
                "type": "LineString",
                "coordinates": [(float(x), float(y)) for x, y in line.coords]
            },
            "properties": {
                "cost_seconds": float(nx.path_weight(G, path, weight="length_m")),
                "nodes": len(path)
            }
        }
    except nx.NetworkXNoPath:
        raise HTTPException(status_code=404, detail="No path found")

In [109]:
def _build_from_shapefile(path: str):
    global G, NODE_XY, NODE_IDS, TREE, GEODETIC_TO_TARGET
    GEODETIC_TO_TARGET = _build_transformer()
    gdf = _load_shapefile(path)
    G = _gdf_to_graph(gdf)
    NODE_XY, NODE_IDS, TREE = _prepare_kdtree(G)
_build_from_shapefile(DATA_SHP)

Building road graph: 100%|██████████| 117005/117005 [00:26<00:00, 4384.76feat/s]


In [110]:
# nid = list(NODE_IDS)
# print(TREE)
# nid.index([515378.0 3774863.0])

In [111]:
u_id, _ = _nearest_node_id( 51.446113342128164, 35.67395123492202)
v_id, _ = _nearest_node_id(51.44679998785106, 35.682770890714394)
print(u_id)
feature = _route(u_id, v_id)
print(feature)

[ 540366.931 3947887.055]
[ 540428.981 3948863.363]
[ 540366.931 3947887.055]


NodeNotFound: Source [ 540366.931 3947887.055] is not in G